In [ ]:

# ████████████████████████████████████████████████████████████████████████████
# ██ SECCIÓN 1: IMPORTS Y CONFIGURACIÓN
# ████████████████████████████████████████████████████████████████████████████
import os
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import logging

try:
    import brian2
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "brian2", "-q"])
    import brian2

from brian2 import (NeuronGroup, Synapses, SpikeMonitor,
                    Network, mV, ms, Hz, second)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')


# ████████████████████████████████████████████████████████████████████████████
# ██ SECCIÓN 2: CARGADOR DEL CONECTOMA
# ████████████████████████████████████████████████████████████████████████████
class FlyWireConnectomeLoader:
    def __init__(self, dataset='fafb', version='v783', api_token=None, data_dir='./flywire_data'):
        self.dataset = dataset
        self.version = version
        self.api_token = api_token or os.getenv('FLYWIRE_API_TOKEN')
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)
        self.meta_file = self.data_dir / f"{dataset}_{version}_meta.json"
        self.base_url = "https://codex.flywire.ai/api/download"
        self.synapse_data_product = "fafb_v783_princeton_synapse_table"

    def _get_file_hash(self, filepath):
        hasher = hashlib.sha256()
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hasher.update(chunk)
        return hasher.hexdigest()

    def _load_local_meta(self):
        if self.meta_file.exists():
            with open(self.meta_file, 'r') as f:
                return json.load(f)
        return None

    def _save_local_meta(self, meta):
        with open(self.meta_file, 'w') as f:
            json.dump(meta, f)

    def check_for_updates(self):
        local_meta = self._load_local_meta()
        remote_meta = {'version': self.version, 'hash': 'simulated_remote_hash_' + self.version}
        if local_meta is None:
            return True
        if local_meta.get('hash') != remote_meta.get('hash'):
            return True
        return False

    def download_data(self, force=False):
        local_meta = self._load_local_meta()
        if not force and local_meta and not self.check_for_updates():
            logging.info("Usando datos locales existentes.")
            return self.data_dir / f"{self.synapse_data_product}.csv"

        logging.info("Generando datos sintéticos del conectoma (modo prueba)...")
        self._create_synthetic_data_file()

        new_meta = {
            'version': self.version,
            'hash': self._get_file_hash(self.data_dir / f"{self.synapse_data_product}.csv"),
            'download_date': pd.Timestamp.now().isoformat()
        }
        self._save_local_meta(new_meta)
        return self.data_dir / f"{self.synapse_data_product}.csv"

    def _create_synthetic_data_file(self):
        n_synapses = 20000
        data = {
            'id': np.arange(n_synapses),
            'pre_pt_root_id': np.random.randint(1, 2000, n_synapses),
            'post_pt_root_id': np.random.randint(1, 2000, n_synapses),
            'connection_score': np.random.uniform(0, 1, n_synapses),
            'cleft_score': np.random.uniform(0, 1, n_synapses),
            'neuropil': np.random.choice(['ME_L', 'ME_R', 'LO_L', 'LO_R'], n_synapses)
        }
        df = pd.DataFrame(data)
        filepath = self.data_dir / f"{self.synapse_data_product}.csv"
        df.to_csv(filepath, index=False)
        logging.info(f"Datos sintéticos creados en {filepath}")

    def load_connections(self):
        filepath = self.data_dir / f"{self.synapse_data_product}.csv"
        if not filepath.exists():
            logging.error("El archivo de datos no existe. Ejecute download_data primero.")
            return None
        df = pd.read_csv(filepath)
        logging.info(f"Cargadas {len(df)} conexiones.")
        return df


# ████████████████████████████████████████████████████████████████████████████
# ██ SECCIÓN 3: CONSTRUCTOR DEL MODELO LIF
# ████████████████████████████████████████████████████████████████████████████
def build_lif_neuron_from_data(connections_df, neuron_id):
    incoming = connections_df[connections_df['post_pt_root_id'] == neuron_id]
    outgoing = connections_df[connections_df['pre_pt_root_id'] == neuron_id]

    params = {
        'neuron_id': neuron_id,
        'v_0': -52 * mV,
        'v_rst': -52 * mV,
        'v_th': -45 * mV,
        't_mbr': 20 * ms,
        'tau': 5 * ms,
        't_rfc': 2.2 * ms,
        'w_syn': 0.275 * mV,
        'n_incoming': len(incoming),
        'n_outgoing': len(outgoing),
    }

    logging.info(f"Neurona {neuron_id}: {len(incoming)} entradas, {len(outgoing)} salidas")
    return params


# ████████████████████████████████████████████████████████████████████████████
# ██ SECCIÓN 4: BUCLE CEREBRO <-> CUERPO (con feedback negativo)
# ████████████████████████████████████████████████████████████████████████████
def simulate_brain_body_closed_loop(connections_df,
                                     n_sensory=50, n_motor=10,
                                     duration_s=5.0,
                                     base_input_rate=60 * Hz,
                                     target_height=1.0,
                                     k_feedback=300 * Hz,
                                     thrust_per_spike=0.05):
    """
    Bucle cerrado con feedback NEGATIVO hacia una altura objetivo.

    Física del cuerpo:
        Cada spike motor añade 'thrust_per_spike' m/s de velocidad vertical.
        Gravedad estándar (-9.81 m/s²).
        Resolución de física: 1 sub-paso por chunk (10 ms).

    Controlador:
        nueva_tasa = base + k_feedback * (target_height - y)
        - Si y < target: tasa sube -> más empuje -> sube
        - Si y > target: tasa baja -> menos empuje -> baja
        - Si y = target: tasa = base -> equilibrio
    """
    np.random.seed(42)

    counts = connections_df.groupby('post_pt_root_id').size().sort_values(ascending=False)
    neuron_id = int(counts.index[0])
    params = build_lif_neuron_from_data(connections_df, neuron_id)

    eqs = '''
    dv/dt = (v_0 - v + g) / t_mbr : volt (unless refractory)
    dg/dt = -g / tau : volt (unless refractory)
    '''

    # ---- Capa sensorial: generador Poisson con tasa dinámica ----
    sensory = NeuronGroup(n_sensory,
                          'rate : Hz',
                          threshold='rand() < rate * dt',
                          method='exact')
    sensory.rate = base_input_rate

    # ---- Capa de procesamiento ----
    brain = NeuronGroup(1, eqs,
                        threshold='v > v_th',
                        reset='v = v_rst; g = 0*mV',
                        refractory=params['t_rfc'],
                        method='exact', namespace=params)
    brain.v = params['v_0']; brain.g = 0 * mV

    # ---- Capa motora ----
    motor = NeuronGroup(n_motor, eqs,
                        threshold='v > v_th',
                        reset='v = v_rst; g = 0*mV',
                        refractory=params['t_rfc'],
                        method='exact', namespace=params)
    motor.v = params['v_0']; motor.g = 0 * mV

    # ---- Sinapsis sensorial -> brain ----
    syn_sens_brain = Synapses(sensory, brain, model='w : volt', on_pre='g += w')
    syn_sens_brain.connect()
    umbral_efectivo = params['v_th'] - params['v_0']
    target_g_ss = umbral_efectivo * 1.4
    R_total = n_sensory * base_input_rate
    syn_sens_brain.w = target_g_ss / (R_total * params['tau'])

    # ---- Sinapsis brain -> motor ----
    syn_brain_motor = Synapses(brain, motor, model='w : volt', on_pre='g += w')
    syn_brain_motor.connect()
    t_mbr_val, tau_val = 20.0, 5.0
    a, b = 1.0 / t_mbr_val, 1.0 / tau_val
    t_peak = np.log(b / a) / (b - a)
    peak_factor = (a / (a - b)) * (np.exp(-b * t_peak) - np.exp(-a * t_peak))
    syn_brain_motor.w = (umbral_efectivo / peak_factor) * 1.3

    mon_sensory = SpikeMonitor(sensory)
    mon_brain = SpikeMonitor(brain)
    mon_motor = SpikeMonitor(motor)

    net = Network(sensory, brain, motor,
                  syn_sens_brain, syn_brain_motor,
                  mon_sensory, mon_brain, mon_motor)

    # ---- Estado del cuerpo ----
    G = -9.81                        # m/s²
    y, vy = target_height, 0.0       # empezamos en la altura objetivo

    brain_dt = 10 * ms
    dt_chunk = brain_dt / second     # 0.01 s por chunk
    n_chunks = int(duration_s * 1000 / (brain_dt / ms))

    motor_spike_count = 0
    trace = {'t': [], 'y': [], 'vy': [], 'rate': [], 'motor': []}

    logging.info(f"Bucle cerrado: {duration_s}s, {n_chunks} chunks de {brain_dt}")

    for chunk in range(n_chunks):
        # 1) Avanzar el cerebro 10 ms
        net.run(brain_dt)

        # 2) Contar spikes motores nuevos en este chunk
        nuevos = mon_motor.num_spikes - motor_spike_count
        motor_spike_count = mon_motor.num_spikes

        # 3) FÍSICA CORREGIDA: cada spike = un impulso de velocidad
        vy += nuevos * thrust_per_spike   # impulso de los motores
        vy += G * dt_chunk                # gravedad
        y  += vy * dt_chunk               # posición
        if y <= 0:
            y, vy = 0.0, 0.0              # suelo

        # 4) FEEDBACK NEGATIVO: error respecto a la altura objetivo
        error = target_height - y
        nueva_tasa = base_input_rate + k_feedback * error
        if nueva_tasa < 0 * Hz:
            nueva_tasa = 0 * Hz           # no puede haber tasa negativa
        sensory.rate = nueva_tasa

        # 5) Muestrear
        trace['t'].append(chunk * (brain_dt / ms))
        trace['y'].append(y)
        trace['vy'].append(vy)
        trace['rate'].append(nueva_tasa / Hz)
        trace['motor'].append(nuevos)

    # ---- Resultados ----
    print("\n" + "=" * 55)
    print("BUCLE CEREBRO <-> CUERPO (feedback negativo)")
    print("=" * 55)
    print(f"Capa sensorial : {mon_sensory.num_spikes} disparos")
    print(f"Capa conectoma : {mon_brain.num_spikes} disparos "
          f"({mon_brain.num_spikes/duration_s:.1f} Hz)")
    print(f"Capa motora    : {mon_motor.num_spikes} disparos")
    print(f"\nCuerpo:")
    print(f"  Altura inicial : {target_height:.2f} m")
    print(f"  Altura final   : {y:.3f} m")
    print(f"  Altura mínima  : {min(trace['y']):.3f} m")
    print(f"  Altura máxima  : {max(trace['y']):.3f} m")
    print(f"  Altura promedio: {np.mean(trace['y']):.3f} m")
    print(f"  Desv. estándar : {np.std(trace['y']):.3f} m")
    print(f"\nÚltimos 10 muestreos:")
    print(f"  t(ms)   y(m)    vy(m/s)  rate(Hz) motor")
    for i in range(-10, 0):
        print(f"  {trace['t'][i]:>5.0f}  {trace['y'][i]:>6.3f}  {trace['vy'][i]:>7.3f}  "
              f"{trace['rate'][i]:>7.1f}  {trace['motor'][i]:>5}")

    return trace


# ████████████████████████████████████████████████████████████████████████████
# ██ SECCIÓN 5: MAIN
# ████████████████████████████████████████████████████████████████████████████
def main():
    print("=" * 60)
    print("BUCLE CEREBRO <-> CUERPO (CONECTOMA FLYWIRE)")
    print("=" * 60)

    loader = FlyWireConnectomeLoader(
        dataset='fafb', version='v783',
        api_token=os.getenv('FLYWIRE_API_TOKEN', 'TU_TOKEN_AQUI'),
        data_dir='./flywire_test_data'
    )
    loader.download_data(force=False)
    connections = loader.load_connections()
    if connections is None:
        logging.error("Sin datos. Abortando.")
        return

    trace = simulate_brain_body_closed_loop(
        connections,
        n_sensory=50,
        n_motor=10,
        duration_s=5.0,          # 5 segundos
        base_input_rate=60 * Hz,
        target_height=1.0,       # queremos mantener 1 m
        k_feedback=300 * Hz,     # ganancia del controlador
        thrust_per_spike=0.05,   # 0.05 m/s por spike motor
    )

    print("\n" + "=" * 60)
    print("RESUMEN")
    print("=" * 60)
    y_final = trace['y'][-1]
    y_avg = np.mean(trace['y'])
    if abs(y_avg - 1.0) < 0.15:
        print(f"  ✅ Estabilizado en {y_avg:.2f} m (objetivo 1.00 m)")
    elif y_avg > 0.5:
        print(f"  ⚠️  Parcialmente funcional: {y_avg:.2f} m")
    else:
        print(f"  ❌ Se cayó al suelo: {y_avg:.2f} m")


if __name__ == "__main__":
    main()